# 1.1 Schema discovery

**What this notebook is for.** Before anything is trained, somebody has to answer
a boring question: do the three parquet datasets this project reads actually have
the columns we think they have, and do those columns have the same type in all
three? They have not always. This notebook answers that question and writes the
answer to S3 so the rest of the pipeline can read it instead of re-deriving it.

**The three datasets.** `trust_score_05.ml.discovery` names them as constants:

- `DATASET_TRAINING_NONFRAUD` — the customers we train on. Assumed innocent,
  which is the whole premise of an unsupervised approach: the model learns what
  normal looks like, and fraud is whatever fails to look like it.
- `DATASET_TESTING_NONFRAUD` — a held-out slab of the same population, used as
  the negative class at evaluation time.
- `DATASET_FRAUD` — the confirmed-fraud records. Small. Used only at evaluation
  time, never in training.

**What "discovery" produces.** Two things. A *schema*: for each dataset, the
column list with dtypes, plus which columns are missing from which dataset. And a
*quality report*: a list of `Check` objects, each with a severity of
`SEVERITY_INFO`, `SEVERITY_WARNING` or `SEVERITY_ERROR`. An error means do not
proceed; a warning means proceed and write it down.

**Run this notebook when** the upstream lineage job has produced a new snapshot,
or when a feature was added to the feature library, or when a training run fails
in a way that smells like a missing column. It is cheap and it is the first thing
to check.

**What it costs.** It reads the parquet *footers*, not the data, so it is minutes
not hours. The one expensive option is `null_rates=True`, which does scan.

In [ ]:
import os
import sys
import warnings

warnings.filterwarnings("ignore")

# The notebooks live two levels below the repo root. Put the root on sys.path so
# `import trust_score_05` resolves without an editable install, which is what
# happens on EMR and SageMaker where the package arrives as a zip.
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print("repo root:", REPO_ROOT)

## Configuration

`load_ml_config` merges YAML overlays from `conf/ml/` in order, so `base.yaml`
holds everything shared and `dev.yaml` / `sandbox.yaml` / `local.yaml` hold only
the differences. The `env` argument picks the overlay. Nothing in this notebook
hardcodes a bucket or a path — if you find yourself wanting to, add a field to
`MLConfig` instead, because a path typed into a notebook is a path that will be
wrong in three weeks and nobody will know where to look.

In [ ]:
from trust_score_05.ml.config import load_ml_config

ENV = "dev"  # one of: local, dev, sandbox

cfg = load_ml_config(env=ENV)

print("env             :", ENV)
print("base path       :", cfg.base_path)
print("join key        :", cfg.join_key)
print("scopes          :", cfg.scopes)
print("models          :", cfg.models)

In [ ]:
from trust_score_05.ml.jobs.base import build_spark_session

spark = build_spark_session("notebook")
spark.sparkContext.setLogLevel("WARN")
spark

## Where the data is

`discover_*_paths` do the globbing. They return `DiscoveredPaths`, which carries
both the paths it found and the prefixes it looked under, so a surprising result
is diagnosable without re-reading the config.

The fraud paths are the ones to look at hardest. Fraud arrives from several
sources with different naming conventions, and a name mismatch here does not
raise — it silently produces an empty fraud population, and then every
evaluation metric comes back as a perfect score against zero positives. That
exact bug is written up in the ML findings document. Check the count.

In [ ]:
from trust_score_05.ml.datasets import (
    discover_fraud_paths,
    discover_testing_nonfraud_paths,
    discover_training_nonfraud_paths,
)

train_paths = discover_training_nonfraud_paths(cfg)
test_paths = discover_testing_nonfraud_paths(cfg)
fraud_paths = discover_fraud_paths(cfg)

for name, found in [
    ("training non-fraud", train_paths),
    ("testing non-fraud", test_paths),
    ("fraud", fraud_paths),
]:
    print(f"{name:22s} {len(found.paths):3d} path(s)")
    for path in found.paths[:5]:
        print(f"{'':22s}   {path}")
    if len(found.paths) > 5:
        print(f"{'':22s}   ... and {len(found.paths) - 5} more")

assert fraud_paths.paths, (
    "No fraud paths found. This does NOT raise downstream -- it produces an "
    "empty fraud population and perfect-looking metrics. Fix the config before "
    "going further."
)

## Run discovery

`run_discovery` collects the schema and runs the checks in one call. Set
`null_rates=True` only when you want the null-rate warnings; it triggers a full
scan of every column in every dataset, which on the real snapshot is the
difference between two minutes and forty.

In [ ]:
from trust_score_05.ml.discovery import run_discovery

report = run_discovery(spark, cfg, null_rates=False)

print("datasets    :", sorted(report.schema))
print("checks      :", len(report.checks))
print("worst       :", report.severity)

## Read the checks

Sort by severity and read every error before reading anything else. `Check`
carries a `name`, a `severity`, a `message` and a `details` dict; the details are
where the actual column names are, so print them for anything that is not
`SEVERITY_INFO`.

In [ ]:
from trust_score_05.ml.discovery import SEVERITY_ERROR, SEVERITY_INFO, SEVERITY_WARNING

ORDER = {SEVERITY_ERROR: 0, SEVERITY_WARNING: 1, SEVERITY_INFO: 2}

for check in sorted(report.checks, key=lambda c: ORDER[c.severity]):
    print(f"[{check.severity:7s}] {check.name}")
    print(f"            {check.message}")
    if check.severity != SEVERITY_INFO and check.details:
        for key, value in check.details.items():
            shown = value if not isinstance(value, list) else value[:10]
            print(f"            - {key}: {shown}")
    print()

## The two failure modes worth understanding

**Partial columns.** `partial_columns` lists columns present in some datasets and
absent from others. This is normal and expected — the fraud feed carries case
metadata the non-fraud feed has no reason to have. It matters because
`add_missing_and_select` in `datasets.py` fills the absentees with typed nulls so
the frames union cleanly, and a column that is *supposed* to be everywhere but
shows up here is a real bug upstream.

**Dtype disagreements.** `dtype_disagreements` lists columns whose type differs
between datasets. This one is never benign. A count that is `bigint` in training
and `string` in fraud will union to `string`, and every model will then treat it
as unusable and silently drop it — so the feature does not appear in the selected
set and nobody notices the model got worse.

In [ ]:
from trust_score_05.ml.discovery import dtype_disagreements, partial_columns

partial = partial_columns(report.schema)
print(f"{len(partial)} partial column(s)")
for name, present_in in sorted(partial.items())[:20]:
    print(f"  {name:55s} present in {sorted(present_in)}")

print()
disagreements = dtype_disagreements(report.schema)
print(f"{len(disagreements)} dtype disagreement(s)")
for name, types in sorted(disagreements.items()):
    print(f"  {name:55s} {types}")

assert not disagreements, "Fix the dtype disagreements before training on this snapshot."

## The numeric intersection

`common_numeric_columns` is the set the feature selector starts from: numeric in
*every* dataset. It is the honest ceiling on how many features any model in this
project can use, and it is worth watching over time — a sudden drop means
something upstream changed type.

In [ ]:
from trust_score_05.ml.discovery import common_numeric_columns

numeric = common_numeric_columns(report.schema)
print(f"{len(numeric)} numeric columns common to all three datasets")
print()
for name in sorted(numeric)[:40]:
    print(" ", name)
if len(numeric) > 40:
    print(f"  ... and {len(numeric) - 40} more")

## Publish

`write_discovery_reports` writes the schema and the checks under
`schema_prefix(cfg)` and drops a `_READY` marker. Downstream stages read the
marker, not the files, so a half-written report is never mistaken for a complete
one. Pass `failures=True` only if you want the report published even though it
contains errors — normally you do not.

In [ ]:
from trust_score_05.ml.discovery import write_discovery_reports
from trust_score_05.ml.paths import schema_prefix

prefix = schema_prefix(cfg)
print("writing to:", prefix)

# Flip to True to actually write. Left False so that opening this notebook and
# hitting run-all does not overwrite a good report with a stale one.
PUBLISH = False

if PUBLISH:
    written = write_discovery_reports(report, prefix, failures=False)
    print("wrote:", written)
else:
    print("PUBLISH is False -- nothing written.")

## What next

- Errors here? Stop, fix the upstream lineage job, re-run.
- Clean? Go to `1.2-population-and-labels.ipynb` to look at the label side, then
  `2.0-features/2.2-feature-selection.ipynb`.
- Doing this for real rather than exploring? The same thing runs headless as
  `python -m trust_score_05.ml.jobs.discovery_job --env dev`. The notebook exists
  to *look* at the report; the job exists to produce it on a schedule.